In [1]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets transformers sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [2]:
import unsloth
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
import pandas as pd

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.5.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [5]:
df = pd.read_parquet("/content/cleaned_support_dataset.parquet")

df.head()

,Combined Text,Ticket Type,Ticket Subject,Ticket Priority
0,i'm having an issue with the gopro hero. we ap...,technical issue,product setup,critical
1,i'm having an issue with the lg smart tv. if y...,technical issue,peripheral compatibility,critical
2,i'm facing a problem with my dell xps. the del...,technical issue,network problem,low
3,i'm having an issue with the microsoft office....,billing inquiry,account access,low
4,i'm having an issue with the autodesk autocad....,billing inquiry,data loss,low


In [6]:
df = df[[
    "Combined Text",
    "Ticket Type",
    "Ticket Subject",
    "Ticket Priority"
]]

df.head()

,Combined Text,Ticket Type,Ticket Subject,Ticket Priority
0,i'm having an issue with the gopro hero. we ap...,technical issue,product setup,critical
1,i'm having an issue with the lg smart tv. if y...,technical issue,peripheral compatibility,critical
2,i'm facing a problem with my dell xps. the del...,technical issue,network problem,low
3,i'm having an issue with the microsoft office....,billing inquiry,account access,low
4,i'm having an issue with the autodesk autocad....,billing inquiry,data loss,low


In [7]:
# import re

# def clean_text(text):

#     text = str(text)

#     # Remove URLs
#     text = re.sub(r"http\S+|www\S+", "", text)

#     # Remove weird placeholders
#     text = re.sub(r"\{.*?\}", "", text)

#     # Remove version numbers
#     text = re.sub(r"\b\d+(\.\d+)+\b", "", text)

#     # Remove forum junk
#     text = re.sub(r"join date:.*", "", text, flags=re.IGNORECASE)

#     # Remove repeated spaces/newlines
#     text = re.sub(r"\s+", " ", text)

#     return text.strip()

# df["Combined Text"] = df["Combined Text"].apply(clean_text)

In [8]:
for i in range(5):
    print(df["Combined Text"].iloc[i])
    print("\n-------------------\n")

i'm having an issue with the gopro hero. we appreciate that you have requested a . please double check your email address. i've tried troubleshooting steps mentioned in the user manual, but the issue persists. resolution pending

-------------------

i'm having an issue with the lg smart tv. if you need to change an existing product. i'm having an issue with the lg smart tv. if the issue i'm facing is intermittent. sometimes it works fine, but other times it acts up unexpectedly. resolution pending

-------------------

i'm facing a problem with my dell xps. the dell xps is not turning on. it was working fine until yesterday, but now it doesn't respond. 1.8.3 i really i'm using the original charger that came with my dell xps, but it's not charging properly.

-------------------

i'm having an issue with the microsoft office. if you have a problem you're interested in and i'd love to see this happen, please check out the . i've already contacted customer support multiple times, but the 

In [9]:
df = df.dropna()

df["Ticket Type"] = (
    df["Ticket Type"]
    .str.lower()
    .str.replace(" ", "_")
)

df["Ticket Subject"] = (
    df["Ticket Subject"]
    .str.lower()
    .str.replace(" ", "_")
)

df["Ticket Priority"] = (
    df["Ticket Priority"]
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

,Combined Text,Ticket Type,Ticket Subject,Ticket Priority
0,i'm having an issue with the gopro hero. we ap...,technical_issue,product_setup,critical
1,i'm having an issue with the lg smart tv. if y...,technical_issue,peripheral_compatibility,critical
2,i'm facing a problem with my dell xps. the del...,technical_issue,network_problem,low
3,i'm having an issue with the microsoft office....,billing_inquiry,account_access,low
4,i'm having an issue with the autodesk autocad....,billing_inquiry,data_loss,low


In [10]:
EOS_TOKEN = tokenizer.eos_token

def format_example(row):

    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a customer support routing AI.
Return ONLY valid JSON.

<|eot_id|><|start_header_id|>user<|end_header_id|>

Classify this support ticket:

{row['Combined Text']}

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{
    "intent": "{row['Ticket Type']}",
    "priority": "{row['Ticket Priority']}"
}}{EOS_TOKEN}"""

df["text"] = df.apply(format_example, axis=1)

df["text"][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a customer support routing AI.\nReturn ONLY valid JSON.\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nClassify this support ticket:\n\ni\'m having an issue with the gopro hero. we appreciate that you have requested a . please double check your email address. i\'ve tried troubleshooting steps mentioned in the user manual, but the issue persists. resolution pending\n\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{\n    "intent": "technical_issue",\n    "subject": "product_setup",\n    "priority": "critical"\n}<|eot_id|>'

In [11]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["text"]])

dataset

Dataset({
    features: ['text'],
    num_rows: 8469
})

In [12]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

train_dataset, val_dataset

(Dataset({
     features: ['text'],
     num_rows: 7622
 }),
 Dataset({
     features: ['text'],
     num_rows: 847
 }))

In [13]:
import torch

In [14]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    packing = False,

    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5,
        learning_rate = 1e-4,

        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),

        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,

        optim = "adamw_8bit",
        weight_decay = 0.05,
        lr_scheduler_type = "linear",

        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/7622 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/847 [00:00<?, ? examples/s]

In [15]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,622 | Num Epochs = 4 | Total steps = 1,908
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,1.113187,1.061868
100,0.796657,0.744171
150,0.722306,0.711481
200,0.698545,0.696527
250,0.716554,0.689955
300,0.695053,0.681394
350,0.660247,0.676254
400,0.654759,0.674356
450,0.721427,0.672625
500,0.682685,0.670902


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=1908, training_loss=0.6879721442608463, metrics={'train_runtime': 2650.2876, 'train_samples_per_second': 11.504, 'train_steps_per_second': 0.72, 'total_flos': 2.3695996027723776e+16, 'train_loss': 0.6879721442608463, 'epoch': 4.0})

In [16]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [17]:
test_ticket = """
I was charged twice for my subscription payment and I need this fixed immediately.
"""

prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a customer support routing AI.
Return ONLY valid JSON.

<|eot_id|><|start_header_id|>user<|end_header_id|>

Classify this ticket:

{test_ticket}

<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.0,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

Both `max_new_tokens` (=40) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

system

You are a customer support routing AI.
Return ONLY valid JSON.

user

Classify this ticket:


I was charged twice for my subscription payment and I need this fixed immediately.


assistant
{
    "intent": "billing_inquiry",
    "subject": "product_setup",
    "priority": "medium"
}


In [22]:
!zip -r support_router_model.zip support_router_model

  adding: support_router_model/ (stored 0%)
  adding: support_router_model/adapter_config.json (deflated 58%)
  adding: support_router_model/chat_template.jinja (deflated 71%)
  adding: support_router_model/README.md (deflated 65%)
  adding: support_router_model/tokenizer_config.json (deflated 96%)
  adding: support_router_model/adapter_model.safetensors (deflated 8%)
  adding: support_router_model/tokenizer.json (deflated 85%)


In [19]:
model.save_pretrained("support_router_model")
tokenizer.save_pretrained("support_router_model")

Unsloth: Restored added_tokens_decoder metadata in support_router_model/tokenizer_config.json.


('support_router_model/tokenizer_config.json',
 'support_router_model/chat_template.jinja',
 'support_router_model/tokenizer.json')

In [20]:
print(df["Ticket Type"].value_counts())

print(df["Ticket Subject"].value_counts())

print(df["Ticket Priority"].value_counts())

Ticket Type
refund_request          1752
technical_issue         1747
cancellation_request    1695
product_inquiry         1641
billing_inquiry         1634
Name: count, dtype: int64
Ticket Subject
refund_request              576
software_bug                574
product_compatibility       567
delivery_problem            561
hardware_issue              547
battery_life                542
network_problem             539
installation_support        530
product_setup               529
payment_issue               526
product_recommendation      517
account_access              509
peripheral_compatibility    496
data_loss                   491
cancellation_request        487
display_issue               478
Name: count, dtype: int64
Ticket Priority
medium      2192
critical    2129
high        2085
low         2063
Name: count, dtype: int64


In [21]:
for label in df["Ticket Type"].unique():
    print("\n\nLABEL:", label)
    samples = df[df["Ticket Type"] == label]["Combined Text"].head(3)

    for i, s in enumerate(samples):
        print(f"\nExample {i+1}:")
        print(s[:500])



LABEL: technical_issue

Example 1:
i'm having an issue with the gopro hero. we appreciate that you have requested a . please double check your email address. i've tried troubleshooting steps mentioned in the user manual, but the issue persists. resolution pending

Example 2:
i'm having an issue with the lg smart tv. if you need to change an existing product. i'm having an issue with the lg smart tv. if the issue i'm facing is intermittent. sometimes it works fine, but other times it acts up unexpectedly. resolution pending

Example 3:
i'm facing a problem with my dell xps. the dell xps is not turning on. it was working fine until yesterday, but now it doesn't respond. 1.8.3 i really i'm using the original charger that came with my dell xps, but it's not charging properly.


LABEL: billing_inquiry

Example 1:
i'm having an issue with the microsoft office. if you have a problem you're interested in and i'd love to see this happen, please check out the . i've already contacted customer 